# 🏛️ 04 - LE SÉNAT : VOTE FINAL V8 (Lightweight)

Version allégée : Uniquement l'agrégation des CSVs et l'application du règlements.
Pas de ré-entraînement, pas d'audit lourd.

### 🎭 Les Sénateurs (Fichiers CSV attendus) :
1. **Sénateur V1** : `divination_v1_fast_predictions.csv` (Renommez votre meilleur `submission_SNIPER_ELITE.csv` ici)
2. **Sénateur V2** : `divination_v2_predictions.csv` (Le Poisson)
3. **Sénateur V3** : `divination_v3_features_predictions.csv` (Features Engineering)

### 📜 Règle du Vote (Amendement Mariage Frères) :
- **Base** : Majorité Absolue (>= 2 voix).
- **Rattrapage** : Si Vote=NON (1/3) ET **Ancien Client** ET **10-16 pages** => FORCE OUI.

In [1]:
import pandas as pd
import numpy as np
import os

# Chemins attendus
FILES = {
    "V1": "divination_v1_fast_predictions.csv",
    "V2": "divination_v2_predictions.csv",
    "V3": "divination_v3_features_predictions.csv"
}

votes = pd.DataFrame()

print("📥 Chargement des bulletins de vote...")
for name, path in FILES.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        # On suppose que la colonne s'appelle 'converted' ou 'prediction'
        col = 'converted' if 'converted' in df.columns else 'prediction'
        votes[name] = df[col]
        print(f"  ✅ {name} chargé ({df[col].sum()} OUI)")
    else:
        print(f"  ❌ {name} MANQUANT ({path})")

if len(votes.columns) < 3:
    print("\n⚠️ ATTENTION : Il manque des sénateurs !")
else:
    print("\n🔔 Quorum atteint (3/3).")

📥 Chargement des bulletins de vote...
  ✅ V1 chargé (918 OUI)
  ✅ V2 chargé (896 OUI)
  ✅ V3 chargé (885 OUI)

🔔 Quorum atteint (3/3).


## 2. Le Vote & Amendement Mariage Frères

In [2]:
# 1. Vote Majoritaire Simple
votes['Somme'] = votes.sum(axis=1)
votes['Verdict_Base'] = (votes['Somme'] >= 2).astype(int)

# 2. Chargement des données Test pour le rattrapage
test_df = pd.read_csv('conversion_data_test.csv')

# Condition : Vote 1/3 (donc NON) MAIS Profil "Mariage Frères"
condition_rattrapage = (
    (test_df['new_user'] == 0) & 
    (test_df['total_pages_visited'] >= 10) & 
    (test_df['total_pages_visited'] <= 16) &
    (votes['Somme'] == 1)
)

n_rep = condition_rattrapage.sum()
print(f"⛑️ Amendement 'Mariage Frères' : {n_rep} utilisateurs repêchés.")

votes['Verdict_Final'] = votes['Verdict_Base']
votes.loc[condition_rattrapage, 'Verdict_Final'] = 1

⛑️ Amendement 'Mariage Frères' : 17 utilisateurs repêchés.


In [3]:
filename = 'submission_LE_SENAT_V8.csv'
submission = votes[['Verdict_Final']].rename(columns={'Verdict_Final': 'converted'})
submission.to_csv(filename, index=False)

print(f"✅ Soumission prête : {filename}")

✅ Soumission prête : submission_LE_SENAT_V8.csv
